In [1]:
import os
from pathlib  import Path

In [2]:
%pwd

'e:\\ai\\deep learning project\\research'

In [3]:
import os

os.chdir(r"E:\ai\deep learning project")

print("Current Directory:", os.getcwd())

Current Directory: E:\ai\deep learning project


In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [6]:
from Cnnclassifier.constants import *
from Cnnclassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config


In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [9]:
from Cnnclassifier.components.model_training import Training

In [10]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)



    
    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [11]:
import inspect
from Cnnclassifier.components.model_training import Training

print(inspect.getfile(Training))

E:\ai\deep learning project\src\Cnnclassifier\components\model_training.py


In [12]:
print(inspect.getsource(Training.get_base_model))

    def get_base_model(self):
        print("Step 1: Loading model")

        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path,
            compile=False
        )

        print("Step 2: Model loaded")

        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(),
            loss="categorical_crossentropy",
            metrics=["accuracy"]
        )

        print("Step 3: Model compiled")
        print("Optimizer:", type(self.model.optimizer))



In [13]:
config = ConfigurationManager()
training_config = config.get_training_config()

training = Training(training_config)

training.get_base_model()
training.train_valid_generator()
training.train()

[2026-08-03 20:48:08,955: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-03 20:48:08,971: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-03 20:48:08,973: INFO: common: created directory at: artifacts]
[2026-08-03 20:48:08,976: INFO: common: created directory at: artifacts\training]
Step 1: Loading model
Step 2: Model loaded
Step 3: Model compiled
Optimizer: <class 'keras.src.optimizers.adam.Adam'>
Found 1471 images belonging to 2 classes.
Found 5889 images belonging to 2 classes.
368/368 - 759s - 2s/step - accuracy: 0.9695 - loss: 0.0993 - val_accuracy: 0.8482 - val_loss: 0.4172
[2026-08-03 21:00:49,840: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]


In [13]:
training.get_base_model()
print(type(training.model.optimizer))
print(training.model.optimizer)

Step 1: Loading model
Step 2: Model loaded
Step 3: Model compiled
Optimizer: <class 'keras.src.optimizers.adam.Adam'>
<class 'keras.src.optimizers.adam.Adam'>
